# Corporate Intelligence Analyst Agent (LIVE DATA)
## Enterprise Agent Capstone Project

### The Pitch
**Problem:**  
Modern investors are drowning in data. Analyzing a single company requires synthesizing 10-K filings, earnings transcripts, real-time news, and market sentiment. For retail investors and busy professionals, doing this manually for every stock of interest is impossible.

**Solution:**  
The **Corporate Intelligence Analyst** is a multi-agent system that acts as an "Analyst Team in a Box." It autonomously gathers, analyzes, and synthesizes diverse data streams to produce a professional-grade, 3-paragraph investor summary in seconds.

**Value Proposition:**
- **Speed:** Reduces hours of research to seconds.
- **Objectivity:** Data-driven analysis without emotional bias.
- **Synthesis:** Connects the dots between hard numbers (Fundamentals) and soft signals (News/Sentiment).

### Architecture
The system follows a **"Newsroom"** architecture, orchestrated by a Manager Agent using the **Google Agent Development Kit (ADK)**.

```mermaid
graph TD
    User[User Input: Ticker Symbol] --> Manager[Manager Agent: Editor-in-Chief]
    Manager -->|Uses Tool| Quant[Quant Agent: Fundamentals]
    Manager -->|Uses Tool| Investigator[Investigator Agent: Risk & News]
    Manager -->|Uses Tool| Futurist[Futurist Agent: Outlook]
    
    Quant -->|Returns Financial Data| Manager
    Investigator -->|Returns Risk Report| Manager
    Futurist -->|Returns Experimental Forecast| Manager
    
    Manager -->|Synthesizes Final Report| Report[Final 3-Paragraph Summary]
```

### Key Concepts Demonstrated
1.  **Multi-Agent Orchestration:** Using `AgentTool` to enable agents to call other agents.
2.  **Tool Use:** Agents utilizing specific python functions and the built-in `google_search` tool.
3.  **Structured Output:** Enforcing a strict, professional reporting format.

## Setup & Configuration

In [1]:
import os
import logging
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, google_search
from google.genai import types

# Setup basic logging to see tool calls
logging.basicConfig(level=logging.INFO)

# Load API Key from .env file
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ GOOGLE_API_KEY not found in environment. Please add it to your .env file.")
else:
    print("✅ API Key loaded successfully.")

/Users/saroshfarhan/Documents/DataScience/AI_Agent-1/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ API Key loaded successfully.


## Core Tools with Logging
We've added print statements to these tools to help debug exactly what data is being passed around.

In [2]:
import yfinance as yf

def get_fundamentals(ticker: str) -> str:
    """Fetches financial fundamentals for a given ticker using yfinance."""
    print(f"\n[DEBUG] get_fundamentals called for: {ticker}")
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        # Extract key metrics with error handling
        revenue_growth = info.get('revenueGrowth', 'N/A')
        operating_margins = info.get('operatingMargins', 'N/A')
        total_cash = info.get('totalCash', 'N/A')
        total_debt = info.get('totalDebt', 'N/A')
        market_cap = info.get('marketCap', 'N/A')
        pe_ratio = info.get('trailingPE', 'N/A')
        
        # Format the data
        data = (
            f"Market Cap: {market_cap}\n"
            f"Revenue Growth: {revenue_growth}\n"
            f"Operating Margins: {operating_margins}\n"
            f"Total Cash: {total_cash}\n"
            f"Total Debt: {total_debt}\n"
            f"P/E Ratio: {pe_ratio}"
        )
    except Exception as e:
        data = f"Error fetching data: {str(e)}"
        
    print(f"[DEBUG] get_fundamentals returning: {data}")
    return data

def get_outlook(ticker: str) -> str:
    """Generates an experimental short-term outlook."""
    print(f"\n[DEBUG] get_outlook called for: {ticker}")
    # Mock data
    data = "Outlook: Bullish. Confidence: 63%. Reasoning: Strong earnings momentum outweighs regulatory concerns."
    print(f"[DEBUG] get_outlook returning: {data}")
    return data

# Wrapper for google_search to add logging
def search_tool(query: str) -> str:
    """Searches the web for information."""
    print(f"\n[DEBUG] search_tool called for: {query}")
    # Call the actual ADK tool
    # Note: google_search is a tool instance, we might need to invoke it or use it directly.
    # For simplicity in this debug phase, we'll just use it as is in the agent list,
    # but if we want to log it, we'd wrap it. 
    # Let's trust the agent to call 'google_search' directly for now, 
    # but we can inspect the Investigator's output in the test section.
    return "Search functionality active (logging wrapper not fully implemented for built-in tool)"

## Agents Setup (Text Output)
We have reverted to simple text output to ensure stability. We will test each agent individually.

In [5]:
MODEL_NAME = "gemini-2.5-flash"

# 1. Quant Agent
quant_agent = Agent(
    name="QuantAgent",
    model=MODEL_NAME,
    description="A financial analyst that provides fundamental data.",
    instruction="You are a financial analyst. Use the `get_fundamentals` tool to analyze the company's financial health. Return a concise text summary of the fundamentals.",
    tools=[get_fundamentals]
)

# 2. Investigator Agent
investigator_agent = Agent(
    name="InvestigatorAgent",
    model=MODEL_NAME,
    description="A risk investigator that finds news, regulatory issues, and sentiment.",
    instruction="""You are a risk investigator. Use the `google_search` tool to identify risks, regulatory issues, and market sentiment.
    Search for terms like 'lawsuits', 'regulatory investigation', 'scandal', and 'analyst sentiment'.
    Return a concise text summary of the risks and sentiment found.""",
    tools=[google_search]
)

# 3. Futurist Agent
futurist_agent = Agent(
    name="FuturistAgent",
    model=MODEL_NAME,
    description="A market futurist that provides an experimental short-term outlook.",
    instruction="You are a market futurist. Use the `get_outlook` tool to provide an experimental forecast. Return a concise text summary of the outlook.",
    tools=[get_outlook]
)

# 4. Manager Agent (The Orchestrator)
manager_agent = Agent(
    name="ManagerAgent",
    model=MODEL_NAME,
    instruction="""You are the Editor-in-Chief of an investment newsletter.
    Your goal is to produce a comprehensive 3-paragraph report on a company based on input from your team.
    
    Process:
    1. Call `QuantAgent` to get the financial fundamentals.
    2. Call `InvestigatorAgent` to get risks and news.
    3. Call `FuturistAgent` to get the outlook.
    4. Synthesize all information into a final report with exactly these three sections:
       - **Summary** (Fundamentals)
       - **Risks & Recent Developments** (News/Risks)
       - **Experimental Outlook** (Forecast)
    """,
    tools=[
        AgentTool(quant_agent),
        AgentTool(investigator_agent),
        AgentTool(futurist_agent)
    ]
)

## Orchestration Demo

In [6]:
# Run the full system
runner = InMemoryRunner(agent=manager_agent)

# Uncomment to run when API key is set in .env
await runner.run_debug("Analyze Apple Inc. (AAPL)")

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False



 ### Created new session: debug_session_id

User > Analyze Apple Inc. (AAPL)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model:


[DEBUG] get_outlook called for: AAPL
[DEBUG] get_outlook returning: Outlook: Bullish. Confidence: 63%. Reasoning: Strong earnings momentum outweighs regulatory concerns.


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.


...

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False


[DEBUG] get_fundamentals returning: Market Cap: 3951253782528
Revenue Growth: 0.079
Operating Margins: 0.31647
Total Cash: 54697000960
Total Debt: 112377004032
P/E Ratio: 35.64257


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the 

ManagerAgent > **Summary**
Apple Inc. (AAPL) demonstrates robust financial fundamentals, boasting a substantial market capitalization of $3.95 trillion. The company has achieved a solid revenue growth rate of 7.9% and maintains healthy operating margins around 31.65%. While Apple holds a significant cash reserve of $54.70 billion, its total debt stands at $112.38 billion. With a P/E ratio of 35.64, the stock reflects a higher valuation in relation to its current earnings.

**Risks & Recent Developments**
Apple Inc. is navigating a complex landscape of legal and regulatory challenges, including prominent antitrust lawsuits from the U.S. Department of Justice and X Corp/xAI, alongside class-action lawsuits alleging securities fraud related to AI promises and ongoing investigations into data privacy practices. The company also faces several patent infringement disputes concerning its Watch technology and other products, as well as scrutiny over labor practices, supply chain ethics, and en

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Apple Inc. (AAPL) financial fundamentals'
         },
         id='adk-669457a9-a6e6-48e1-8d41-c1f6aa5eab76',
         name='QuantAgent'
       ),
       thought_signature=b'\n\xeb\x04\x01\xd1\xed\x8ao4\x95\x03\xe7\xa1P2-\'\xba\xf4\xb0\xd6Z\x13\xe9|q^u\'"x4\x19[\x8e\xb4\xda\xcdq\x05j\xe4\xd4T\xfe\xdb\xb4\xae\xf8\xd2\xa1 [\x9b\x8aO\xa0M+\xad\xfc\xf0\x84,Z\xb9I\x11`\xae\x8b\x96\x17 \xd6*\xbb\x89\xb7\x8f\x8a\xf0q\xb4\xd8\x05\x8d\xa1,\x93\xa3\x88\x10G\x1e=\xf7...'
     ),
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Apple Inc. (AAPL) risks and recent news'
         },
         id='adk-d61155ac-66ae-45cb-a604-21e9e5bba509',
         name='InvestigatorAgent'
       )
     ),
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Apple Inc. (AAPL) short-term outlook'
     